<a href="https://colab.research.google.com/github/allmore0/pp_501/blob/main/Proyecto_semestre_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Created on Tue Mar 10 19:16:04 2026

@author: Alberto Lozano Morales

Universidad Nacional Rosario Catellanos - Magdalena Contreras
Grupo: 501
"""

'\nCreated on Tue Mar 10 19:16:04 2026\n\n@author: Alberto Lozano Morales\n\nUniversidad Nacional Rosario Catellanos - Magdalena Contreras\nGrupo: 501\n'

In [ ]:
# Módulos importados para el proyecto
import pandas as pd
import requests
import io
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Notas: La información a partir de 2023 es por municipio y no contamos con información de 2025 y 2026
# Proceso 3 importaciones para homologar los layouts de 2003 a 2014, 2015 a 2020 y 2021 a 2024
df_mercado_justo = pd.read_csv('/content/drive/MyDrive/pp_501/Cierre_agricola_mun_2003_2024.csv', encoding='latin1')

# Verificación de carga (Criterio 2 de la rúbrica)
if df_mercado_justo is not None:
    print(df_mercado_justo.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pp_501/Cierre_agricola_mun_2003_2024.csv'

In [ ]:
df_limon_michoacan = df_mercado_justo[(df_mercado_justo['Nomcultivo'] == 'Limón') &
                                      (df_mercado_justo['Nomestado'] == 'Michoacán')]
df_limon_michoacan

In [ ]:
df_limon_michoacan['Anio'].unique()

### **Regresión Lineal y Visualización de Precios del Limón en Michoacán**

Para analizar la tendencia del precio$_1$ del limón en Michoacán a lo largo de los años, realizaremos una regresión lineal.Ajustamos un modelo de regresión lineal simple donde el año de registro será la variable independiente y el precio será la variable dependiente. Finalmente, visualizaremos los datos reales junto con la línea de regresión y una gráfica de líneas del precio promedio anual.
#
#
**precio$_1$:** *Para el limón es el precio medio rural, la unidad de medida son pesos por tonelada, con excepción de los cultivos que tienen otra métrica la cual se señala en el nombre del cultivo (gruesa, manojo, planta, entre otros).*

In [ ]:
# Respaldo para eliminar filas con valores NaN en las columnas relevantes para la regresión
df_limon_michoacan_ok = df_limon_michoacan.dropna(subset=['Precio', 'Anio']).copy()

# Datos para la regresión lineal
X = df_limon_michoacan_ok[['Anio']] # Variable independiente (Año)
y = df_limon_michoacan_ok['Precio']    # Variable dependiente (Precio Final)

# Modelo de regresión lineal
model = LinearRegression()
model.fit(X, y)

# Predicciones con el modelo
y_pred = model.predict(X)

# --- Visualización de Regresión Lineal (Gráfica de Dispersión con Línea de Regresión) ---
plt.figure(figsize=(12, 7))
sns.scatterplot(x='Anio', y='Precio', data=df_limon_michoacan_ok, label='Precio Real', alpha=0.6)
plt.plot(X, y_pred, color='red', linewidth=2, label='Línea de Regresión Lineal')
plt.title('Regresión Lineal del precio del limón en Michoacán (2003-2024)')
plt.xlabel('Año de registro')
plt.ylabel('Precio')
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del Precio Promedio Anual (Gráfica Lineal) ---
plt.figure(figsize=(12, 7))
promedio_precio_por_año = df_limon_michoacan_ok.groupby('Anio')['Precio'].mean().reset_index()
sns.lineplot(x='Anio', y='Precio', data=promedio_precio_por_año, marker='o', color='blue', label='Precio Promedio Anual')
plt.title('Precio promedio anual del limón en Michoacán (2003-2024)')
plt.xlabel('Año de registro')
plt.ylabel('Precio promedio')
plt.legend()
plt.grid(True)
plt.show()

### **Pronóstico del Precio del Limón para 2026 con Random Forests Regressor**

Usamos boques aleatorios *(RandomForestRegressor)*, para predecir el precio del limón en Michoacán de 2026. Es adecuado por la capacidad que tiene el modelo para manejar relaciones no lineales y su robustez. Se entrenara el modelo con los precios promedio anuales calculados anteriormente.

In [ ]:
# Preparar los datos para el modelo de Random Forests
# Usaremos el precio promedio anual para el pronóstico
X_rf = average_price_per_year[['Anio']]
y_rf = average_price_per_year['Precio']

# Crear y ajustar el modelo de Random Forests Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42) # 100 árboles en el bosque
rf_model.fit(X_rf, y_rf)

# Realizar la predicción para el año 2026
anio_a_predecir = pd.DataFrame([[2026]], columns=['Anio'])
precio_2026_rf = rf_model.predict(anio_a_predecir)

print(f"El precio pronosticado del limón para 2026 (Random Forests) es: {precio_2026_rf[0]:,.2f}")

# --- Visualización del Pronóstico ---
plt.figure(figsize=(12, 7))
sns.lineplot(x='Anio', y='Precio', data=average_price_per_year, marker='o', color='blue', label='Precio Promedio Anual Histórico')
plt.scatter(2026, precio_2026_rf[0], color='green', s=100, zorder=5, label=f'Pronóstico 2026 ({precio_2026_rf[0]:,.2f})')
plt.title('Pronóstico del Precio Promedio Anual del Limón en Michoacán (Random Forests)')
plt.xlabel('Año de Registro')
plt.ylabel('Precio Promedio')
plt.legend()
plt.grid(True)
plt.show()

### **Análisis de comportamiento del precio del limón con** $Cadenas de Markov$

La finalidad es entender el *comportamiento* del precio (es decir, si tiende a subir, bajar o mantenerse), por esto usamos la Cadena de Markov. Definiremos **estados** basados en la *- variación anual del precio -* y calcularemos las probabilidades de transición entre estos **estados**.

In [ ]:
import numpy as np

# Calcular la variación anual del precio
price_changes = average_price_per_year.sort_values('Anio').copy()
price_changes['precio_anterior'] = price_changes['Precio'].shift(1)
price_changes['cambio_porcentual'] = (price_changes['Precio'] - price_changes['precio_anterior']) / price_changes['precio_anterior']

# Definir estados de precio (ej. Aumento, Disminución, Estable)
# Podemos ajustar los umbrales según lo que consideremos 'estable'
def get_state(change):
    if change > 0.05:  # Aumento de más del 5%
        return 'Aumento'
    elif change < -0.05: # Disminución de más del 5%
        return 'Disminucion'
    else:
        return 'Estable'

price_changes['estado_precio'] = price_changes['cambio_porcentual'].apply(get_state)

# Eliminar el primer NaN que resulta del shift
price_changes = price_changes.dropna(subset=['estado_precio'])

# Crear la matriz de transición
states = ['Aumento', 'Disminucion', 'Estable']
transition_matrix = pd.DataFrame(0, index=states, columns=states)

for i in range(1, len(price_changes)):
    current_state = price_changes.iloc[i-1]['estado_precio']
    next_state = price_changes.iloc[i]['estado_precio']
    transition_matrix.loc[current_state, next_state] += 1

# Normalizar la matriz para obtener probabilidades
transition_matrix = transition_matrix.div(transition_matrix.sum(axis=1), axis=0)

print("Matriz de Transición de Estados del Precio:")
display(transition_matrix)

# Pronosticar el estado más probable para 2026 (basado en el último estado observado)
last_observed_state = price_changes.iloc[-1]['estado_precio']
most_probable_next_state = transition_matrix.loc[last_observed_state].idxmax()

print(f"\nEl último estado observado del precio fue: {last_observed_state}")
print(f"El estado más probable del precio para el siguiente período (2026) es: {most_probable_next_state}")